# Financial News Sentiment Classification

Sentiment classification on financial text, built on the Financial PhraseBank benchmark. Compares a classical TF-IDF baseline against a fine-tuned DistilBERT transformer and an off-the-shelf domain model (FinBERT), with token-level explainability and a deployable REST API.

Financial language inverts everyday sentiment. "The company cut costs" is positive. "Growth slowed to 8%" is negative despite the growth. General-purpose sentiment models get this wrong constantly, which is what makes the domain worth a dedicated model.

**Run this on a GPU runtime** (Colab: Runtime → Change runtime type → T4). Fine-tuning on CPU works but takes roughly 20x longer.

## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    precision_recall_fscore_support, f1_score
)

sns.set_style("whitegrid")
%matplotlib inline

# seed everything so the numbers are reproducible
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

## 2. Data Loading

Financial PhraseBank — sentences from financial news, each labeled negative / neutral / positive by finance students.

Loaded from the CSV distribution rather than the HuggingFace `datasets` package: the Hub version ships as a loading script, and `datasets` v4 dropped script support. The CSV has no header row, and the file is Latin-1 encoded rather than UTF-8.

In [ ]:
# all-data.csv has no header row - columns are label, then sentence
df = pd.read_csv("all-data.csv", encoding="latin-1", names=["label_name", "sentence"])

label_names = ["negative", "neutral", "positive"]
df["label"] = df["label_name"].apply(lambda x: label_names.index(x))

print("Shape:", df.shape)
df.head()

In [ ]:
# sanity check the mapping - a silent mislabel here poisons every metric downstream
print("Label values :", sorted(df["label"].unique()))
assert set(df["label"].unique()) == {0, 1, 2}, "Unexpected label values"

# neutral should be the majority class in PhraseBank - if it isn't, the mapping is wrong
print(df["label_name"].value_counts())
assert df["label_name"].value_counts().idxmax() == "neutral", "Label order looks wrong"

df.head()

## 3. Exploratory Data Analysis

In [ ]:
df["label_name"].value_counts()

In [ ]:
df["n_words"] = df["sentence"].apply(lambda s: len(s.split()))
df["n_words"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

sns.countplot(data=df, x="label_name", order=label_names, ax=axes[0])
axes[0].set_title("Label Distribution")
axes[0].set_xlabel("")

sns.histplot(data=df, x="n_words", bins=40, ax=axes[1])
axes[1].set_title("Sentence Length (words)")

plt.tight_layout()
plt.show()

In [ ]:
# what does each class actually look like
for name in label_names:
    print("===", name.upper(), "===")
    for s in df[df["label_name"] == name]["sentence"].head(3):
        print("  -", s)
    print()

**Finding:** the classes are heavily imbalanced — neutral dominates. A model that predicts "neutral" for everything would score a deceptively high accuracy while being useless, so **macro-F1 is the metric to watch here, not accuracy**. Same reasoning as weighting attack recall over aggregate accuracy in an intrusion detector.

Sentences are also short (mostly under 40 words), which matters for picking a max token length later.

## 4. Train / Validation / Test Split

Three-way stratified split. The validation set exists so early stopping and any threshold or hyperparameter choice happens **without touching test data** — otherwise the reported score stops being an honest estimate.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train :", len(train_df))
print("Val   :", len(val_df))
print("Test  :", len(test_df))
print()
print(train_df["label_name"].value_counts(normalize=True).round(3))

## 5. Baseline: TF-IDF + Logistic Regression

Establishes the performance floor. If a fine-tuned transformer can't clearly beat bag-of-words, the added complexity isn't earning its cost.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),      # bigrams catch "not profitable", "cut costs"
    min_df=2,
    sublinear_tf=True,
    stop_words=None         # keep stopwords - "not", "no", "down" carry the signal here
)

X_train_tfidf = vectorizer.fit_transform(train_df["sentence"])
X_val_tfidf = vectorizer.transform(val_df["sentence"])
X_test_tfidf = vectorizer.transform(test_df["sentence"])

print("Vocabulary size :", len(vectorizer.vocabulary_))

In [ ]:
logres_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",   # counter the neutral-heavy imbalance
    random_state=SEED
)
logres_model.fit(X_train_tfidf, train_df["label"])

tfidf_preds = logres_model.predict(X_test_tfidf)

print(classification_report(test_df["label"], tfidf_preds, target_names=label_names))

In [ ]:
cm = confusion_matrix(test_df["label"], tfidf_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()
plt.title("TF-IDF + Logistic Regression")
plt.show()

In [ ]:
# which words push each class - a sanity check that the baseline learned finance, not noise
feature_names = np.array(vectorizer.get_feature_names_out())

for i, name in enumerate(label_names):
    coefs = logres_model.coef_[i]
    top = np.argsort(coefs)[::-1][:12]
    print(f"{name:9s} ->", ", ".join(feature_names[top]))

## 6. DistilBERT Fine-Tuning

DistilBERT is a distilled BERT — about 40% smaller and 60% faster, with most of the accuracy retained. Good tradeoff for a dataset this size.

The training loop is written out manually rather than using the `Trainer` API, so every piece (batching, scheduler, gradient clipping, early stopping) is visible and adjustable.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 64        # EDA showed sentences are short - 64 tokens covers nearly all of them
BATCH_SIZE = 32
EPOCHS = 6
PATIENCE = 2
LR = 2e-5           # standard fine-tuning LR - much higher and pretrained weights get wrecked

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer)

In [ ]:
# check the max length choice against the actual token distribution
tok_lens = [len(tokenizer.encode(s)) for s in train_df["sentence"]]
print("Token length - mean:", round(np.mean(tok_lens),1), "| 95th pct:", int(np.percentile(tok_lens,95)), "| max:", max(tok_lens))
print("Sentences truncated at MAX_LEN =", MAX_LEN, ":", sum(1 for l in tok_lens if l > MAX_LEN))

In [ ]:
class FinancialDataset(Dataset):
    def __init__(self, sentences, labels, tokenizer, max_len):
        self.sentences = list(sentences)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.sentences[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
train_ds = FinancialDataset(train_df["sentence"], train_df["label"], tokenizer, MAX_LEN)
val_ds = FinancialDataset(val_df["sentence"], val_df["label"], tokenizer, MAX_LEN)
test_ds = FinancialDataset(test_df["sentence"], test_df["label"], tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print("Batches per epoch :", len(train_loader))

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names)
).to(DEVICE)

# class weights - same imbalance fix as the baseline, applied through the loss
counts = train_df["label"].value_counts().sort_index().values
class_weights = torch.tensor(counts.sum() / (len(counts) * counts), dtype=torch.float).to(DEVICE)
print("Class weights :", class_weights.cpu().numpy().round(3))

loss_fn = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=LR, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),   # warmup stops early batches from wrecking pretrained weights
    num_training_steps=total_steps
)

In [ ]:
def evaluate(model, loader):
    """Run the model over a loader, return (loss, preds, labels)."""
    model.eval()
    losses, all_preds, all_labels = [], [], []

    with torch.inference_mode():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs.logits, labels)

            losses.append(loss.item())
            all_preds.extend(outputs.logits.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.mean(losses), np.array(all_preds), np.array(all_labels)

In [ ]:
best_val_f1 = 0.0
best_state = None
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "val_f1": []}

for epoch in range(EPOCHS):
    bert_model.train()
    running = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["label"].to(DEVICE)

        outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs.logits, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)   # standard for transformer fine-tuning
        optimizer.step()
        scheduler.step()

        running += loss.item()

    train_loss = running / len(train_loader)
    val_loss, val_preds, val_labels = evaluate(bert_model, val_loader)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch} | train {train_loss:.4f} | val {val_loss:.4f} | val macro-F1 {val_f1:.4f}")

    # early stopping on macro-F1, not loss - F1 is what we actually care about
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.detach().cpu().clone() for k, v in bert_model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= PATIENCE:
        print("Early stopping at epoch", epoch)
        break

bert_model.load_state_dict(best_state)
bert_model.to(DEVICE)
print("Restored best model, val macro-F1 :", round(best_val_f1, 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Training Curve")
axes[0].legend()

axes[1].plot(history["val_f1"], marker="o", color="green")
axes[1].axvline(int(np.argmax(history["val_f1"])), color="red", linestyle="--", label="best checkpoint")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Macro-F1")
axes[1].set_title("Validation Macro-F1")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
_, bert_preds, bert_labels = evaluate(bert_model, test_loader)

print(classification_report(bert_labels, bert_preds, target_names=label_names))

In [ ]:
cm = confusion_matrix(bert_labels, bert_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot()
plt.title("DistilBERT (fine-tuned)")
plt.show()

## 7. FinBERT (Zero-Shot)

FinBERT is BERT pretrained further on financial text and already fine-tuned for sentiment. Running it with no training at all is a useful third comparison: it answers "could I have just downloaded a model instead of fine-tuning one?"

In [ ]:
FINBERT_NAME = "ProsusAI/finbert"

finbert_tokenizer = AutoTokenizer.from_pretrained(FINBERT_NAME)
finbert_model = AutoModelForSequenceClassification.from_pretrained(FINBERT_NAME).to(DEVICE)

# FinBERT's label order differs from PhraseBank's - read it from config instead of assuming
print("FinBERT id2label :", finbert_model.config.id2label)
print("PhraseBank order :", label_names)

In [ ]:
# build the index mapping from FinBERT's label order to ours
finbert_to_ours = {}
for idx, name in finbert_model.config.id2label.items():
    name = name.lower()
    if name in label_names:
        finbert_to_ours[int(idx)] = label_names.index(name)

print("mapping :", finbert_to_ours)
assert len(finbert_to_ours) == len(label_names), "Label mapping incomplete - check id2label"

In [ ]:
finbert_model.eval()
finbert_preds = []

with torch.inference_mode():
    for i in range(0, len(test_df), BATCH_SIZE):
        chunk = test_df["sentence"].iloc[i:i+BATCH_SIZE].tolist()
        enc = finbert_tokenizer(chunk, truncation=True, padding=True,
                                max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        logits = finbert_model(**enc).logits
        raw_preds = logits.argmax(dim=1).cpu().numpy()
        finbert_preds.extend([finbert_to_ours[int(p)] for p in raw_preds])

finbert_preds = np.array(finbert_preds)

print(classification_report(test_df["label"], finbert_preds, target_names=label_names))

## 8. Model Comparison

In [ ]:
def score(name, y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    return pd.Series({"Accuracy": acc, "Macro Precision": p, "Macro Recall": r, "Macro F1": f}, name=name)

results = pd.DataFrame([
    score("TF-IDF + LogReg", test_df["label"], tfidf_preds),
    score("FinBERT (zero-shot)", test_df["label"], finbert_preds),
    score("DistilBERT (fine-tuned)", bert_labels, bert_preds),
])

results.round(4)

In [ ]:
ax = results.plot(kind="bar", figsize=(9,4.5))
ax.set_ylabel("Score"); ax.set_ylim(0,1)
ax.set_title("Model Comparison (test set)")
ax.legend(loc="lower right")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# per-class F1 - the macro average hides which class each model struggles with
per_class = pd.DataFrame({
    "TF-IDF": f1_score(test_df["label"], tfidf_preds, average=None, zero_division=0),
    "FinBERT": f1_score(test_df["label"], finbert_preds, average=None, zero_division=0),
    "DistilBERT": f1_score(bert_labels, bert_preds, average=None, zero_division=0),
}, index=label_names)

per_class.round(3)

**Finding:** read the per-class table before trusting the headline number. The minority classes (negative, positive) are where the models actually differ — a big macro-F1 gap driven entirely by the majority neutral class would mean much less.

## 9. Error Analysis

Aggregate metrics say *how much* a model is wrong. This says *where*, which is the part that tells you what to fix.

In [ ]:
errors = test_df.copy().reset_index(drop=True)
errors["pred"] = bert_preds
errors["pred_name"] = errors["pred"].apply(lambda x: label_names[x])
errors["correct"] = errors["label"] == errors["pred"]

print("Test accuracy :", round(errors["correct"].mean(), 4))
print()
print("Most common confusions:")
wrong = errors[~errors["correct"]]
print(wrong.groupby(["label_name", "pred_name"]).size().sort_values(ascending=False).head(6))

In [ ]:
# get the model's confidence so we can find the confidently-wrong cases
bert_model.eval()
all_probs = []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        logits = bert_model(input_ids=input_ids, attention_mask=attention_mask).logits
        all_probs.append(torch.softmax(logits, dim=1).cpu().numpy())

all_probs = np.vstack(all_probs)
errors["confidence"] = all_probs.max(axis=1)

In [ ]:
# confidently wrong = the most informative failures
confident_wrong = errors[~errors["correct"]].sort_values("confidence", ascending=False)

for _, row in confident_wrong.head(8).iterrows():
    print(f"[true: {row['label_name']:8s} | pred: {row['pred_name']:8s} | conf: {row['confidence']:.2f}]")
    print("  ", row["sentence"])
    print()

In [ ]:
plt.figure(figsize=(7,4))
sns.histplot(data=errors, x="confidence", hue="correct", bins=30, element="step")
plt.title("Prediction Confidence: Correct vs Incorrect")
plt.xlabel("Max softmax probability")
plt.show()

**Finding:** a well-calibrated model should be less confident when it's wrong. If the incorrect predictions sit at the same confidence as correct ones, the softmax score can't be trusted as a filter — which matters directly for deployment, since a confidence threshold is the usual way to route uncertain cases to a human.

## 10. Explainability

Which words actually drove a prediction? This uses **occlusion**: remove one token at a time, re-run the model, and measure how far the predicted class probability falls. A big drop means that token was carrying the prediction.

Simple, model-agnostic, and no extra dependencies — the same role SHAP played in the intrusion detection project.

In [ ]:
def explain(sentence, top_n=8):
    """Occlusion-based token attribution for a single sentence."""
    bert_model.eval()
    words = sentence.split()

    def predict_proba(text):
        enc = tokenizer(text, truncation=True, padding="max_length",
                        max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            logits = bert_model(**enc).logits
        return torch.softmax(logits, dim=1).cpu().numpy()[0]

    base = predict_proba(sentence)
    pred_class = int(base.argmax())

    drops = []
    for i in range(len(words)):
        occluded = " ".join(words[:i] + words[i+1:])
        p = predict_proba(occluded)
        drops.append(base[pred_class] - p[pred_class])

    order = np.argsort(drops)[::-1][:top_n]

    print(f"Sentence : {sentence}")
    print(f"Predicted: {label_names[pred_class]}  (confidence {base[pred_class]:.3f})")
    print("-" * 60)
    for i in order:
        direction = "supports" if drops[i] > 0 else "opposes  "
        print(f"  {words[i]:22s} {drops[i]:+.4f}  {direction}")
    print()

In [ ]:
# one from each class
for name in label_names:
    sample = test_df[test_df["label_name"] == name]["sentence"].iloc[0]
    explain(sample)

In [ ]:
# and one the model got confidently wrong - why did it fail?
if len(confident_wrong):
    print("=== CONFIDENTLY WRONG ===")
    explain(confident_wrong.iloc[0]["sentence"])

## 11. Saving the Model

Persisting the fine-tuned model, tokenizer and label order so the classifier can be reloaded without retraining.

In [ ]:
SAVE_DIR = "financial_sentiment_model"

bert_model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# label order has to travel with the model or a reload will silently mislabel everything
with open(f"{SAVE_DIR}/label_names.json", "w") as f:
    json.dump(label_names, f)

print("Saved to", SAVE_DIR)
print(os.listdir(SAVE_DIR))

In [ ]:
# smoke test - reload from disk and check it predicts identically to the in-memory model
reloaded_tok = AutoTokenizer.from_pretrained(SAVE_DIR)
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(DEVICE)
reloaded_model.eval()

check = test_df["sentence"].head(50).tolist()

def batch_predict(model, tok, texts):
    enc = tok(texts, truncation=True, padding="max_length",
              max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.inference_mode():
        return model(**enc).logits.argmax(dim=1).cpu().numpy()

in_memory = batch_predict(bert_model, tokenizer, check)
from_disk = batch_predict(reloaded_model, reloaded_tok, check)

print("Round-trip agreement :", (in_memory == from_disk).mean())
assert (in_memory == from_disk).all(), "Saved model doesn't match in-memory model"
print("Smoke test passed.")

## Conclusions

1. **FinBERT's lead is not a fair win.** FinBERT was itself fine-tuned on Financial PhraseBank, so a random test split of this benchmark overlaps its training data. Its higher score reflects memorisation as much as capability, and it should be read as a contaminated reference point rather than a baseline that was beaten. The clean comparison here is DistilBERT against TF-IDF.

2. **Fine-tuning bought roughly +0.12 macro-F1 over bag-of-words**, and the gain is concentrated in the minority classes rather than spread evenly — which is where a model actually earns its cost on imbalanced data.

3. **Accuracy is the wrong headline metric here.** The neutral class dominates, so a majority-class predictor scores well while being useless. Macro-F1 and the per-class breakdown are what actually describe performance.

4. **The baseline is not a formality.** TF-IDF with bigrams is a genuinely strong floor on short, formulaic financial sentences. A transformer has to clearly beat it to justify the training cost and inference latency.

5. **Confidently-wrong predictions are the useful failures.** They show where the model has learned a shortcut rather than the meaning — and the confidence histogram says whether a softmax threshold could safely route uncertain cases to a human reviewer.

## Known Limitations

- **Dataset size and label noise.** ~4.8k sentences, including annotations where the labellers did not fully agree. More data than the all-agree subset, but noisier labels.
- **Contaminated comparison.** The FinBERT result cannot be interpreted as a fair benchmark without re-testing both models on a corpus FinBERT was not trained on.
- **Sentence-level, not document-level.** Real financial text arrives as full articles or filings where sentiment is mixed and context spans sentences.
- **Annotator bias.** Labels come from finance students judging text in isolation, without market context.
- **No market validation.** This measures agreement with human sentiment labels, *not* whether the sentiment predicts returns. That requires timestamped, ticker-linked headlines aligned to prices point-in-time — a separate project, and the place where look-ahead bias does the most damage.

## Next Steps

- Re-run the FinBERT comparison on a held-out financial corpus outside its training data, to separate memorisation from capability
- Compare against full BERT and RoBERTa to quantify what distillation costs
- Document-level modeling over full articles rather than isolated sentences
- Calibration (temperature scaling) so confidence scores can be trusted as a routing threshold
- Link headlines to tickers and timestamps, then test whether sentiment carries any predictive signal on returns under a strictly time-ordered split